# Quiz 3：泛化、Kernel 與 XAI

沿用 Quiz 2 選定設定，僅訓練集加入溫和 affine / 灰階擾動，不翻轉左右。

以相同 fold 與訓練規則比較；最終有／無擴增重訓使用相同 epoch 數。

In [ ]:
from pathlib import Path
import sys
import subprocess
import json
from IPython.display import display, Image, Markdown
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'Quiz01_DatasetPreparation').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Please open this notebook inside the week3 directory.')
sys.path.insert(0, str(ROOT))
config = json.loads((ROOT / 'Quiz01_DatasetPreparation/Output/stage1_config.json').read_text(encoding='utf-8'))
device = 'cpu'
print('Project:', ROOT, '| Python:', sys.executable, '| device:', device)
display(config)

def run_script(script, *arguments):
    subprocess.run([sys.executable, str(ROOT / script), *map(str, arguments)], check=True, cwd=ROOT)

def show_file(path):
    path = ROOT / path
    if not path.exists():
        print('尚未產生：', path)
    elif path.suffix == '.csv':
        display(pd.read_csv(path))
    elif path.suffix == '.png':
        display(Image(filename=str(path)))
    elif path.suffix == '.md':
        display(Markdown(path.read_text(encoding='utf-8')))
    else:
        display(json.loads(path.read_text(encoding='utf-8')))


In [ ]:
RUN_PREVIEW = False
if RUN_PREVIEW:
    run_script('Quiz03_Generalization/Code/preview_augmentation.py')
show_file('Quiz03_Generalization/Output/augmentation_examples.png')

## 擴增實驗

RUN_CV=True 啟動 10 次擴增 CV；RUN_FINAL=True 啟動兩次重訓與 test。先完成 Quiz 2 CV。

In [ ]:
RUN_CV = False
RUN_FINAL = False
if RUN_CV:
    run_script('Quiz03_Generalization/Code/run.py', '--stage', 'cv', '--device', str(device))
if RUN_FINAL:
    run_script('Quiz03_Generalization/Code/run.py', '--stage', 'final', '--device', str(device))

## 兩個 Kernel 與 Grad-CAM

必須先有正式訓練 checkpoint 與 test 預測。解釋預測類別，呈現各類正確／錯誤案例。

In [ ]:
RUN_EXPLAIN = False
RUN_REPORT = False
if RUN_EXPLAIN:
    for quiz in (2, 3):
        run_script('Quiz03_Generalization/Code/explain.py', '--quiz', str(quiz), '--device', str(device))
if RUN_REPORT:
    run_script('report.py')

In [ ]:
show_file('Quiz03_Generalization/Output/all_models_comparison.csv')
show_file('Quiz03_Generalization/Output/augmentation_comparison.png')
show_file('Quiz03_Generalization/Output/augmentation_analysis.md')
show_file('Quiz03_Generalization/Output/explanations/quiz3/plain/kernels_and_responses.png')
show_file('Quiz03_Generalization/Output/explanations/quiz3/plain/kernel_analysis.md')
for name in ('plain', 'vgg19'):
    show_file(f'Quiz03_Generalization/Output/explanations/quiz3/{name}/xai_analysis.md')
    for path in sorted((ROOT / f'Quiz03_Generalization/Output/explanations/quiz3/{name}').glob('gradcam_test_*.png'))[:4]:
        display(Image(filename=str(path)))

## 結果解讀

檢視擴增後 Top-1、Top-5、Macro-AUC 的變化；沒有提升也需如實分析。

Kernel 需結合權重與實際反應判斷；Grad-CAM 為粗略關注區域，原圖低解析度且無分割標註，不能宣稱精確器官定位。